# TODO:

Politics / Mentions: two promising wallets
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8	
0x0cb10c40b0776e9ee8cef970af85724654dda76c


# Wallet Strategy Selection

Unified selection stage: runs **all wallet-selection methods** and saves each
as a named `WalletSelectionStrategy` artifact in the workspace.  The backtest
stage loads these artifacts directly — no strategy logic is re-built there.

## Methods included
| id suffix | description |
|-----------|-------------|
| `skill_sweep` → `quality_core` | top-N by best skill metric (grid-searched on train-a→train-b) |
| `cohort_early_entry` | wallets that enter markets early |
| `cohort_smooth_pnl` | wallets with high PnL relative to volatility |
| `volatility` | volatility-filtered wallets (from the profitable_wallet_analysis path) |

Each method produces **three trigger variants**:
- `score_threshold`  — composite signal score ≥ calibrated threshold (Kelly sizing)
- `all_open_buys`    — every open-buy event (fixed stake baseline)
- `copy_triggers`    — every trade (open/add/close/reduce), tight slippage


In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

## Configuration

In [2]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [3]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-03-15  TRAIN_A_END_DATE=2025-11-23


## Compute / load wallet skill metrics

In [4]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,10964
train_b_wallets,10964
full_train_wallets,10964
test_wallets,10964


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [5]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
10,avg_copy_roi_capped,50,50,1813,0.07,0.12,1.52,103284.02,0.42
11,avg_copy_roi_capped,100,100,2695,0.07,0.10,1.12,143225.93,0.42
0,prob_edge_shrunk,50,50,4204,0.01,0.12,0.93,115130.45,0.47
22,roi_sharpe,200,200,3250,0.01,0.07,0.85,101101.91,0.70
12,avg_copy_roi_capped,200,200,5272,0.06,0.11,0.82,248959.70,0.48
1,prob_edge_shrunk,100,100,5418,0.03,0.11,0.78,201409.43,0.51
13,avg_copy_roi_capped,300,300,7046,0.06,0.07,0.76,384078.84,0.50
23,roi_sharpe,300,300,4467,0.01,0.07,0.70,117444.18,0.70
6,weighted_prob_edge_shrunk,100,100,2399,0.06,0.09,0.68,256835.63,0.47
21,roi_sharpe,100,100,2473,0.01,0.03,0.66,73063.49,0.78


In [6]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'avg_copy_roi_capped', 'best_top_n': 50}

## Select wallets (skill path) + build cohorts

## Build wallet profiles and signal events

In [7]:
# from polymarket_analysis.signal.builder import (
#     build_wallet_profiles,
#     build_signal_events,
# )

# selected_wallet_profiles = build_wallet_profiles(
#     dataset, selected_wallets, period="full_train",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
# )
# selected_wallet_profiles.to_parquet(
#     WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet", index=False
# )

# # Force-regenerate signal caches: the schema now includes all event types
# # (open_buy, add_buy, close_sell, reduce_sell) plus copy_price/copy_market_key/
# # copy_token_winner columns. Delete stale parquets so they are always rebuilt.
# CALIBRATION_SIGNALS_PATH.unlink(missing_ok=True)
# TEST_SIGNALS_PATH.unlink(missing_ok=True)

# _, train_b_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="train_b",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# train_b_signals.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

# _, test_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="test",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# test_signals.to_parquet(TEST_SIGNALS_PATH, index=False)

# print(f"train_b signals: {len(train_b_signals):,}  test signals: {len(test_signals):,}")
# print(f"event types: {train_b_signals['event_type'].value_counts().to_dict()}")

## Calibrate signal scoring on train-B

In [8]:
# from polymarket_analysis.signal.scorer import (
#     build_calibration_tables,
#     apply_signal_score,
#     select_signal_threshold,
# )

# # Calibration tables are built from open_buy events only — the scoring model
# # is designed for open-buy conviction features. Non-open-buy rows get neutral
# # defaults and will not be used to calibrate the scorer.
# train_b_open_buys = train_b_signals[train_b_signals["event_type"] == "open_buy"].copy()

# price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
# train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
# SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
# print(f"Global edge: {global_edge:.4f}")
# print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# # score distribution
# score_deciles = (
#     train_b_scored.assign(
#         score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop")
#     )
#     .groupby("score_decile")
#     .agg(
#         count=("signal_score", "size"),
#         avg_signal_score=("signal_score", "mean"),
#         avg_copy_roi_capped=("copy_roi_capped", "mean"),
#     )
#     .reset_index()
# )
# score_deciles

## Build wallet cohorts (skill path)

In [9]:
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

# wallet_cohorts = build_wallet_cohorts(
#     full_train_metrics, train_b_open_buys, selected_wallets,
#     top_n=BEST_TOP_N,
# )
# {name: len(df) for name, df in wallet_cohorts.items()}


## Volatility-based wallet selection (second method)

Loads the full training set, applies the volatility filter, and stores the
result as an additional `WalletSelectionStrategy`.  The volatility wallet set
is added to `wallet_cohorts` so the strategy factory handles it uniformly.

In [10]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
    ].copy().reset_index(drop=True)


# Load full training trades for the volatility path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)

df_full = df_full.merge(mdf, on="condition_id", how="inner")

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']

df_full["dt"] = pd.to_datetime(df_full["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_full.columns and "quantity" not in df_full.columns:
    df_full = df_full.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_full["usdc_amount"]      = df_full["usdc_amount"].astype(float)
df_full["final_value_usdc"] = df_full["final_value_usdc"].astype(float)
df_full["quantity"]         = df_full["quantity"].astype(float)

df_full["pnl"] = np.where(
    df_full["side"] == "BUY",
    df_full["final_value_usdc"] - df_full["usdc_amount"],
    df_full["usdc_amount"] - df_full["final_value_usdc"],
)
df_full["notional"] = np.where(
    df_full["side"] == "BUY",
    df_full["usdc_amount"],
    df_full["quantity"] * (1 - df_full["price"].astype(float)),
)
# ignore buys with large price, so we don't skew roi
# df_full = df_full[df_full['price'] <= 0.95]
df_slice = df_full[df_full["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
print(len(wallet_vol_train))
# del df_full, df_slice

17204


In [11]:
wallet_vol_train['copyable_pnl_factor'] = np.clip(wallet_vol_train['copyable_pnl'] / wallet_vol_train['total_pnl'], 0, 1.0)
wallet_vol_train['copyable_roi'] = wallet_vol_train['average_roi'] * wallet_vol_train['copyable_pnl_factor']
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)
# wallet_vol_train = wallet_vol_train[wallet_vol_train['copyable_roi'] > 0.05]
# wallet_vol_train = wallet_vol_train[wallet_vol_train['top_market_pnl_pct'] < 0.3]

In [12]:
wallet_cohorts = {}

In [13]:
print(len(wallet_vol_train))
wallet_vol_train.head()


17204


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
0,0x4b1d2d479f67a33141d67005acf069e10a08bdfb,NaN,1,1,1.52,150.48,150.48,1.00,1.00,99.00,2026-02-27 06:10:00+00:00,99.00,0.00,0.00,0.00,0.00,99.00,1.00,99.00
1,0x868ee2f41837f6df7a5be58d21af5ec62197e14a,12.33,21,1,2111.84,9381.72,2227.22,1.00,1.00,1.02,2026-03-09 06:45:00+00:00,222.21,44.82,0.00,0.00,0.00,4.44,0.24,52.75
2,0x62b61c25cd18c81d15867f62a6d708aff526e575,5.72,2,1,159.38,56.00,56.00,1.00,1.00,49.00,2025-06-03 10:57:30+00:00,49.00,157.23,2.81,157.23,2.81,0.35,1.00,49.00
3,0xaae65d0ad8b93cb5a66aad5dfd80f1c19eb8963f,17.10,2,1,27.50,1805.83,1805.83,1.00,1.00,40.67,2026-02-28 13:17:30+00:00,40.67,5.50,0.00,5.50,0.00,65.67,1.00,40.67
4,0xbba9a972d6f3e8fb79792dad96fe186b55f59051,NaN,1,1,1.00,124.00,39.68,1.00,1.00,124.00,2025-11-04 22:25:00+00:00,124.00,0.00,0.00,0.00,0.00,124.00,0.32,39.68


In [14]:
# filtered_wallets_vol = filter_wallets_by_volatility(
#     wallet_vol_train,
#     min_buckets=20,
#     max_top5_pnl_pct=.6,
#     max_top_market_pnl_pct=0.5,
# )

# filtered_wallets_vol = wallet_vol_train.copy()

# filtered_wallets_vol = (
#     filtered_wallets_vol[
#         (filtered_wallets_vol['average_roi'] >= 0.04)
#         & (filtered_wallets_vol['copyable_pnl'] / filtered_wallets_vol['total_pnl'] >= 0.5)
# ].sort_values("pnl_volatility", ascending=True).head(100)
# )

# print(f"Volatility-filtered wallets: {len(filtered_wallets_vol)}")

# # Build wallet_quality based on total_pnl rank (higher = better)
# filtered_wallets_vol = filtered_wallets_vol.copy().reset_index(drop=True)
# filtered_wallets_vol["wallet_quality"] = filtered_wallets_vol["total_pnl"].rank(
#     method="first", pct=True
# )

# Add as additional cohort (uses only train-b signals for trigger calibration)
# We intersect with wallets that have signals to ensure coverage
# vol_wallets_with_signals = set(train_b_signals["wallet"]).intersection(
#     set(filtered_wallets_vol["wallet"])
# )
# [
#     filtered_wallets_vol["wallet"].isin(vol_wallets_with_signals)
# ][["wallet", "wallet_quality"]].copy().reset_index(drop=True)

In [15]:
wallet_vol_train.sort_values("copyable_pnl", ascending=False).head(10)

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
6421,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,20.42,655,42,3638656.20,1304494.87,501531.92,0.45,0.33,0.27,2025-10-27 08:45:00+00:00,0.35,62723.87,0.05,23756.96,0.05,0.36,0.38,0.13
6296,0xbaa2bcb5439e985ce4ccf815b4700027d1b92c73,14.00,15630,544,15615669.28,716737.58,363530.98,0.73,0.35,0.03,2025-09-21 17:25:00+00:00,0.28,190507.64,0.27,131564.40,0.36,0.05,0.51,0.14
16570,0x1caa6a7ad0c6916aef7b67946de2e57ad24846a0,57.61,139,7,413183.27,463433.04,336778.77,0.99,1.08,-1.00,2026-03-15 06:45:00+00:00,-0.41,36405.69,0.08,10974.77,0.03,1.12,0.73,-0.30
4086,0x88c4919de76e526d55a32c1f8afb439dd1f1129a,7.31,577,56,1360862.67,613998.46,315659.88,0.48,0.41,0.69,2026-03-06 01:00:00+00:00,0.68,24707.51,0.04,7172.25,0.02,0.45,0.51,0.35
4231,0x37d10ffb61998561c5f9fb941c42c952d8fb4e28,23.70,308,22,1245482.91,638367.59,309546.15,0.57,0.47,0.15,2025-06-30 10:25:00+00:00,0.68,40520.75,0.06,40520.75,0.13,0.51,0.48,0.33
4569,0x2a21fec355312b609b9ac97ce4eda3e93ec3d307,18.78,61,17,1559733.52,268316.85,240110.13,0.86,0.30,0.11,2025-11-01 07:50:00+00:00,0.32,21166.73,0.08,21001.44,0.09,0.17,0.89,0.29
3456,0x1f1d38f54a615d1c43c77e3b109bd01f56e163d9,20.33,577,17,769614.27,688774.49,222966.54,0.57,0.66,-1.00,2025-06-20 20:30:00+00:00,1.39,114228.78,0.17,47935.04,0.21,0.89,0.32,0.45
2835,0x09d3273fa76282ce09f4f35a87d6f087c05f4e84,30.12,131,5,187963.16,339257.67,210797.18,0.86,0.96,-1.00,2026-02-18 03:25:00+00:00,0.97,55683.98,0.16,11076.04,0.05,1.80,0.62,0.60
11711,0xc6587b11a2209e46dfe3928b31c5514a8e33b784,11.75,1980,205,31447096.93,925345.26,210699.60,0.41,0.14,0.01,2025-04-01 03:10:00+00:00,-0.05,106917.99,0.12,62446.04,0.30,0.03,0.23,-0.01
5393,0x4dfd481c16d9995b809780fd8a9808e8689f6e4a,103.33,164,44,1746506.21,700981.16,205544.78,1.14,0.33,0.32,2026-01-15 02:02:30+00:00,0.70,348289.16,0.50,228045.36,1.11,0.40,0.29,0.21


In [16]:
# wallet_cohorts['multi_filter'] = wallet_vol_train[
#     (wallet_vol_train['copyable_roi'] >= 0.04)
#     & (wallet_vol_train['num_buckets'] >= 20)
#     &(wallet_vol_train['copyable_pnl'] <= wallet_vol_train['total_pnl'])
#     &(wallet_vol_train['copyable_pnl_factor'] >=0.4)
#     # & (wallet_vol_train['median_roi'] >= 0)
#     & (wallet_vol_train['max_drawdown_to_pnl'] <= 0.3)
#     & (wallet_vol_train['max_copyable_drawdown_to_copyable_pnl'] <= 0.4)
#     # & (wallet_vol_train['num_markets'] >= 100)
#     & (wallet_vol_train['top_market_pnl_pct'] < 0.4)
#     & (wallet_vol_train['top5_pnl_pct'] < 0.5)
#     # & (wallet_vol_train['total_pnl'] < 10_000)
# ].sort_values("copyable_roi", ascending=False).copy().reset_index(drop=True)

wallet_cohorts['multi_filter'] = wallet_vol_train[
    #(wallet_vol_train['copyable_roi'] >= 0.02)
    (wallet_vol_train['average_roi'] >= 0.05)
    # & (wallet_vol_train['copyable_pnl'] <= wallet_vol_train['total_pnl'])
    & (wallet_vol_train['max_drawdown_to_pnl'] <= 0.3)
    # & (wallet_vol_train['max_drawdown_to_pnl'] >= 0.1)
    # & (wallet_vol_train['max_copyable_drawdown_to_copyable_pnl'] <= 0.4)
    & (wallet_vol_train['num_buckets'] >= 20)
    & (wallet_vol_train['top_market_pnl_pct'] < 0.5)
    & (wallet_vol_train['top5_pnl_pct'] < 0.5)
    # & (wallet_vol_train['copyable_pnl'] >= 100)
    # & (wallet_vol_train['copyable_pnl_factor'] >= 0.1)
    # & (wallet_vol_train['total_pnl'] > 10_000)
].sort_values("copyable_roi", ascending=False).copy().reset_index(drop=True)

len(wallet_cohorts['multi_filter'])

131

In [17]:
wallet_vol_train.head()

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
0,0x4b1d2d479f67a33141d67005acf069e10a08bdfb,NaN,1,1,1.52,150.48,150.48,1.00,1.00,99.00,2026-02-27 06:10:00+00:00,99.00,0.00,0.00,0.00,0.00,99.00,1.00,99.00
1,0x868ee2f41837f6df7a5be58d21af5ec62197e14a,12.33,21,1,2111.84,9381.72,2227.22,1.00,1.00,1.02,2026-03-09 06:45:00+00:00,222.21,44.82,0.00,0.00,0.00,4.44,0.24,52.75
2,0x62b61c25cd18c81d15867f62a6d708aff526e575,5.72,2,1,159.38,56.00,56.00,1.00,1.00,49.00,2025-06-03 10:57:30+00:00,49.00,157.23,2.81,157.23,2.81,0.35,1.00,49.00
3,0xaae65d0ad8b93cb5a66aad5dfd80f1c19eb8963f,17.10,2,1,27.50,1805.83,1805.83,1.00,1.00,40.67,2026-02-28 13:17:30+00:00,40.67,5.50,0.00,5.50,0.00,65.67,1.00,40.67
4,0xbba9a972d6f3e8fb79792dad96fe186b55f59051,NaN,1,1,1.00,124.00,39.68,1.00,1.00,124.00,2025-11-04 22:25:00+00:00,124.00,0.00,0.00,0.00,0.00,124.00,0.32,39.68


In [18]:
wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False).head(20)

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
20,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,20.42,655,42,3638656.20,1304494.87,501531.92,0.45,0.33,0.27,2025-10-27 08:45:00+00:00,0.35,62723.87,0.05,23756.96,0.05,0.36,0.38,0.13
5,0x88c4919de76e526d55a32c1f8afb439dd1f1129a,7.31,577,56,1360862.67,613998.46,315659.88,0.48,0.41,0.69,2026-03-06 01:00:00+00:00,0.68,24707.51,0.04,7172.25,0.02,0.45,0.51,0.35
9,0x8861f0bb5e0c19474ba73beeadc13ed8915beed6,16.03,325,17,1361177.08,556342.53,129122.41,0.48,0.45,0.37,2025-05-29 11:35:00+00:00,1.05,25732.59,0.05,499.11,0.00,0.41,0.23,0.24
66,0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,5.38,3478,549,9872547.92,356035.39,116014.93,0.22,0.12,0.01,2025-07-25 03:30:00+00:00,0.11,56515.67,0.16,35000.28,0.30,0.04,0.33,0.04
54,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,10.90,5378,1046,3387993.42,278882.00,72383.65,0.40,0.08,0.06,2025-08-19 10:00:00+00:00,0.18,67467.21,0.24,35112.36,0.49,0.08,0.26,0.05
4,0xc0a005750061e6b5c5699c395a09fd238116ac10,6.56,86,5,135539.35,137409.27,55244.26,0.36,0.41,0.88,2025-06-18 08:12:30+00:00,1.23,0.00,0.00,0.00,0.00,1.01,0.40,0.49
37,0xdfe3fedc5c7679be42c3d393e99d4b55247b73c4,9.91,1834,161,2472889.12,299573.49,52980.57,0.19,0.19,0.25,2025-09-05 16:50:00+00:00,0.42,34988.85,0.12,5532.21,0.10,0.12,0.18,0.07
81,0x5739ddf8672627ce076eff5f444610a250075f1a,4.69,2252,94,2259574.03,254193.40,48981.69,0.15,0.13,0.15,2025-07-24 11:35:00+00:00,0.13,46039.12,0.18,34489.51,0.70,0.11,0.19,0.02
67,0x134a63b764ac7b008356e8db1857db94e6b09e42,4.07,5478,2030,2194458.70,161054.87,41658.87,0.36,0.27,0.01,2025-08-08 20:05:00+00:00,0.14,13607.25,0.08,3247.68,0.08,0.07,0.26,0.04
18,0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,11.46,809,228,974325.25,143703.70,41599.15,0.47,0.11,0.04,2025-12-09 05:20:00+00:00,0.54,13561.54,0.09,10306.51,0.25,0.15,0.29,0.16


In [19]:
df_slice.head(1)

,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,...,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional
0,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d366982...,2025-01-11 15:56:11+00:00,BUY,1633.00,1633.00,0.87,1418.29,0.00,-1418.29,...,-0.00,0.00,2025-01-20T00:00:00Z,Will Biden pardon Adam Schiff?,"[Politics, Biden, Hunter, Creators, zerohedge,...",Politics,1105087383876246719926465389756757310118391264...,No,-1418.29,1418.29


In [20]:
wallets = set(wallet_cohorts['multi_filter']['wallet'])

df = df_full[~df_full["is_train"]].copy()
print(len(df))

df_wallets = df[
    (df['wallet'].isin(wallets))
    & ~df['is_train']
    ].copy()
print(len(df_wallets))

df = df_wallets.groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="pnl", key=np.abs, ascending=False).merge(mdf, on="condition_id", how="left")

len(df)
df[
    (~df['tags'].apply(lambda x: 'Iran' in x))
    & (df['end_date_iso'] > '2026-04-01')
     & (df['primary_tag'] == 'Politics')
   ].sort_values(by='wallets', ascending=False)

1329411
83309


,condition_id,pnl,notional,sell_count,buy_count,wallets,end_date_iso,question,tags,primary_tag,winner_token_id,outcome
10,0x48fbf70c1713e71a405052bc4641e26dbba435fa5576...,19338.67,296799.21,400,461,25,2026-04-30T00:00:00Z,Will Trump visit China by April 30?,"[Politics, Trump, China, TikTok, Trump Preside...",Politics,8119751488573156981734738722995533949652899651...,No
22,0x7434b22007745d99095c102119fdb6b975d34869212e...,10311.32,545643.12,223,385,23,2026-04-30T00:00:00Z,"Russia x Ukraine ceasefire by April 30, 2026?","[Politics, Ukraine, zelensky, Geopolitics, For...",Politics,3250981126890718961468799221688844406693629937...,No
143,0xf30579393c6067a81c801530f7b398ca8eb3491a14a0...,1068.03,32149.82,0,85,13,2026-04-01T00:00:00Z,"Will Trump say ""Crypto"" or ""Bitcoin"" during Ad...","[Politics, Trump, Mentions]",Politics,7854238175616923184908110705304650076400183757...,No
7,0xd8423c8dc3174c6839b0cf667ce1a1364f8532573940...,24685.39,15958.27,30,30,12,2026-05-01T00:00:00Z,"Will Trump say ""American Dream"" at The Village...","[Politics, Trump, Mentions]",Politics,3272224061045573364041762599268785229554066680...,No
2167,0x77791e343f0c9befba6344921e846af705e5b63937c4...,10.66,14898.25,50,106,12,2026-04-30T00:00:00Z,Will Trump talk to Ursula von der Leyen in April?,"[Politics, Trump, Geopolitics]",Politics,6507829190467782246383136963203546034503863357...,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...
623,0x53e409b11e90081ba2cb0834a391e1208dd4c87ac817...,150.10,4949.90,1,0,1,2026-12-31T00:00:00Z,US strike on Mexico by March 31?,"[Politics, Trump, Venezuela, Geopolitics, Fore...",Politics,3968551358101597382806506111501456431970573233...,No
1468,0x3f15706f26c2bf9eeb3c9410683172dde94bb57aa6f8...,29.56,176.62,3,8,1,2026-04-22T00:00:00Z,"Will Starmer say ""Tolerance"" during the next P...","[Politics, Mentions, Starmer, pmq]",Politics,5284335108196272928219691156931808916073389384...,No
1469,0x42d42b30124ed2d93800358dfd1d48253114e4d58cff...,29.56,562.94,0,3,1,2026-04-30T00:00:00Z,Will Trump talk to Nicolás Maduro in April?,"[Politics, Trump, Geopolitics]",Politics,6289234268079812487521700039255987130196834033...,No
1471,0x8fcf7ddd8dca9e5cefdcd107cba81e737bcf1638ad28...,-29.48,74.90,2,6,1,2026-04-19T00:00:00Z,"Will Trump post ""Free Tina Peters"" this week o...","[Politics, Trump, Mentions]",Politics,5498205695403845616679359458716298595322613185...,No


In [21]:
MARKET = '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'

market_def = mdf.loc[mdf['condition_id'] == MARKET].head(1).squeeze()
print(f"Market: {market_def}")

dfc = df_full[(df_full['condition_id'] == MARKET)
              & (df_full['wallet'].isin(wallets))].copy()
# dfc['bucket'] = dfc['dt'].dt.floor('1h')


dfc = dfc.groupby(['dt', 'wallet', 'side', 'outcome']).agg( 
        pnl=('pnl', 'sum'),
        notional=('notional', 'sum'),
        quantity=('quantity', 'sum'),
        position=('position', 'last'),
        avg_price=('price', lambda x: (x * dfc.loc[x.index, 'quantity']).sum() / dfc.loc[x.index, 'quantity'].sum()),
        copyable_qty=('copyable_qty', 'sum'),
        copyable_pnl=('copyable_pnl', 'sum'),
)[['pnl', 'position', 'notional', 'quantity', 'avg_price', 'copyable_qty', 'copyable_pnl']].reset_index().sort_values(['dt', 'wallet', 'side', 'outcome'])

Market: condition_id       0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...
end_date_iso                                    2026-01-31T00:00:00Z
question                       US strikes Iran by February 28, 2026?
tags               [Politics, Iran, Middle East, Geopolitics, Wor...
primary_tag                                                 Politics
winner_token_id    1107900031214423651268558640767076860146505232...
outcome                                                          Yes
Name: 59688, dtype: object


In [22]:
dfc[(dfc['wallet'] == '0xc02147dee42356b7a4edbb1c35ac4ffa95f61fa8')
    & (dfc['quantity'] >= 1) ]['pnl']

Series([], Name: pnl, dtype: float64)

In [23]:
# ── positions per wallet + sum, dual-axis (position + price) ──────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(dfc) != 0:

    # ── 1. build per-bucket price series for YES (volume-weighted avg)
    _yes = dfc[dfc['outcome'] == 'Yes'].copy()
    _yes['_wv'] = _yes['avg_price'] * _yes['quantity']
    price_ts = (
        _yes.groupby('dt')[['_wv', 'quantity']].sum()
        .rename(columns={'quantity': '_qty'})
        .assign(vwap=lambda d: d['_wv'] / d['_qty'])[['vwap']]
        .reset_index()
        .sort_values('dt')
    )

    # ── 2. net YES position per wallet per timestamp (YES pos - NO pos)
    # position is cumulative after each trade; take last value per (dt, wallet, outcome)
    pos_per_wallet = (
        dfc
        .groupby(['dt', 'wallet', 'outcome'])['position']
        .last()
        .reset_index()
    )
    pos_yes = pos_per_wallet[pos_per_wallet['outcome'] == 'Yes'].rename(columns={'position': 'pos_yes'})[['dt', 'wallet', 'pos_yes']]
    pos_no = pos_per_wallet[pos_per_wallet['outcome'] == 'No'].rename(columns={'position': 'pos_no'})[['dt', 'wallet', 'pos_no']]
    net_pos = (
        pos_yes
        .merge(pos_no, on=['dt', 'wallet'], how='outer')
        .fillna(0)
    )
    net_pos['net'] = net_pos['pos_yes'] - net_pos['pos_no']
    net_pos = net_pos.sort_values(['wallet', 'dt']).reset_index(drop=True)

    # ── 3. weight wallets by proper probabilistic score (Brier skill) with shrinkage
    wallet_scores = (
        train_b_metrics[['wallet', 'brier_skill', 'open_buy_trades']]
        .drop_duplicates('wallet')
        .copy()
    )
    wallet_scores['score'] = wallet_scores['brier_skill'].clip(lower=0.0).fillna(0.0)
    wallet_scores['confidence'] = (
        wallet_scores['open_buy_trades'].fillna(0.0)
        / (wallet_scores['open_buy_trades'].fillna(0.0) + 25.0)
    )
    wallet_scores['wallet_weight'] = wallet_scores['score'] * wallet_scores['confidence']
    wallet_weights = wallet_scores.set_index('wallet')['wallet_weight']

    # ── 4. normalize by each wallet's typical move size in this market
    net_pos['net_step'] = net_pos.groupby('wallet')['net'].diff()
    typical_move = net_pos.groupby('wallet')['net_step'].apply(
        lambda s: s.abs().dropna().median() if s.notna().any() else np.nan
    ).rename('typical_move')
    positive_moves = typical_move[typical_move > 0]
    fallback_move = positive_moves.median() if not positive_moves.empty else 1.0
    typical_move = typical_move.fillna(fallback_move).clip(lower=fallback_move * 0.25)

    # ── 5. carry positions forward so the signal reflects current wallet stance
    timeline = price_ts['dt'].sort_values().drop_duplicates()
    net_panel = (
        net_pos.pivot(index='dt', columns='wallet', values='net')
        .sort_index()
        .reindex(timeline)
        .ffill()
        .fillna(0.0)
    )
    available_wallets = net_panel.columns.intersection(wallet_weights.index).intersection(typical_move.index)
    net_panel = net_panel[available_wallets]
    wallet_weights = wallet_weights.reindex(available_wallets).fillna(0.0)
    typical_move = typical_move.reindex(available_wallets)
    normalized_panel = net_panel.divide(typical_move, axis=1)
    weighted_panel = normalized_panel.multiply(wallet_weights, axis=1)

    signal_ts = pd.DataFrame({
        'dt': net_panel.index,
        'total_net': net_panel.sum(axis=1).to_numpy(),
        'weighted_position': weighted_panel.sum(axis=1).to_numpy(),
    })

    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=['YES token (positive weighted signal => predicts YES)'],
        specs=[[{'secondary_y': True}]],
    )

    COLORS = [
        '#4878CF', '#6ACC65', '#D65F5F', '#B47CC7', '#C4AD66',
        '#77BEDB', '#F7A35C', '#90ED7D', '#8085E9', '#F15C80',
    ]

    top_wallets = net_panel.abs().max().sort_values(ascending=False).head(10).index

    # ── per-wallet net position (step: hold latest value, no interpolation)
    for c_idx, wallet in enumerate(top_wallets):
        wdf = (
            net_panel[[wallet]].reset_index()
            .rename(columns={wallet: 'net'})
            .sort_values('dt')
        )
        fig.add_trace(
            go.Scatter(
                x=wdf['dt'],
                y=wdf['net'],
                mode='lines',
                name=f'{wallet[:8]}...',
                line=dict(color=COLORS[c_idx % len(COLORS)], width=1, shape='hv'),
                legendgroup=wallet,
                opacity=0.6,
            ),
            row=1, col=1, secondary_y=False,
        )

    # ── total net position line (primary y, bold)
    fig.add_trace(
        go.Scatter(
            x=signal_ts['dt'],
            y=signal_ts['total_net'],
            mode='lines',
            name='SUM (net YES)',
            line=dict(color='#222222', width=3, shape='hv'),
            legendgroup='sum_net',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── weighted normalized wallet position (primary y)
    fig.add_trace(
        go.Scatter(
            x=signal_ts['dt'],
            y=signal_ts['weighted_position'],
            mode='lines',
            name='Weighted normalized position',
            line=dict(color='#C44E52', width=3, shape='hv'),
            legendgroup='weighted_signal',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── YES price line (secondary y)
    fig.add_trace(
        go.Scatter(
            x=price_ts['dt'],
            y=price_ts['vwap'],
            mode='lines+markers',
            name='Price (YES)',
            line=dict(color='#888888', width=2, dash='dot', shape='hv'),
            marker=dict(size=4),
            legendgroup='price_yes',
        ),
        row=1, col=1, secondary_y=True,
    )

    fig.add_hline(y=0, line=dict(color='#BBBBBB', width=1, dash='dash'))
    fig.update_yaxes(title_text='Net position / weighted normalized total', row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text='Price (USDC)', row=1, col=1, secondary_y=True, range=[0, 1])

    fig.update_layout(
        title=f'Wallet net YES positions and weighted signal - {MARKET[:16]}...',
        height=500,
        template='plotly_white',
        legend=dict(orientation='v', x=1.05),
    )
    fig.show(renderer='browser')


In [24]:
# dominant tags
wallet_fills = df_full[df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet'])]

dominant_tags = (
    wallet_fills
    .groupby(['wallet', 'primary_tag'], as_index=False)[['quantity', 'pnl']].sum()
    .sort_values(['wallet', 'quantity'], ascending=[True, False])
    .assign(total_qty=lambda df: df.groupby('wallet')['quantity'].transform('sum'))
    .drop_duplicates('wallet')
    .assign(primary_tag_ratio=lambda df: df['quantity'] / df['total_qty'])
    .rename(columns={
        'quantity': 'primary_tag_qty'
    })[['wallet', 'primary_tag', 'primary_tag_qty', 'primary_tag_ratio', 'pnl']]
)

In [25]:
print(len(dominant_tags))
dominant_tags.sort_values('pnl', ascending=False).head(20)

131


,wallet,primary_tag,primary_tag_qty,primary_tag_ratio,pnl
272,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,Esports,5739369.43,1.00,1304494.87
1457,0x88c4919de76e526d55a32c1f8afb439dd1f1129a,Politics,3189754.39,0.79,652181.55
1433,0x8861f0bb5e0c19474ba73beeadc13ed8915beed6,Politics,3128613.65,1.00,556118.41
1109,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,Politics,8253939.34,0.87,321354.86
934,0x5739ddf8672627ce076eff5f444610a250075f1a,Politics,3834039.35,0.96,237734.79
1447,0x889e7f0464c72eb8cda1525ebc12b6aaba9d09e0,Politics,11197428.09,0.59,232714.67
1178,0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,Politics,12682462.67,0.66,230517.17
2111,0xd1acd3925d895de9aec98ff95f3a30c5279d08d5,Politics,11053173.87,0.83,166486.74
1065,0x67327ef4880d4050ead195ca23203c61d8c1d7f0,Politics,496100.34,0.69,164689.83
252,0x134a63b764ac7b008356e8db1857db94e6b09e42,Politics,3351730.71,0.78,145069.18


In [26]:
# from polymarket_analysis.data.tags import load_tag_map
# from polymarket_analysis.data.tags import dominant_tag_stats

# split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

# _result = dominant_tag_stats(
#     df_trades=df_full[df_full['dt'] >= split_dt],
#     wallets=wallet_cohorts["multi_filter"]["wallet"],
# )

# print(f"Wallets: {len(_result)}")
# _result[_result['primary_tag'] == 'Politics'].head(5)


In [27]:
# _result.groupby('primary_tag').agg(
#     tag_pnl=('tag_pnl', 'sum'),
#     wallets=('wallet', 'nunique')
#     )

In [28]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False)['wallet'].head(10).to_list() + ['0xdfe3fedc5c7679be42c3d393e99d4b55247b73c4']
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [29]:
print(len(wallet_cohorts['multi_filter']))
pd.set_option('display.max_rows', 100)
wallet_cohorts["multi_filter"].sort_values("total_pnl", ascending=False).head(30)

131


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
20,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,20.42,655,42,3638656.20,1304494.87,501531.92,0.45,0.33,0.27,2025-10-27 08:45:00+00:00,0.35,62723.87,0.05,23756.96,0.05,0.36,0.38,0.13
5,0x88c4919de76e526d55a32c1f8afb439dd1f1129a,7.31,577,56,1360862.67,613998.46,315659.88,0.48,0.41,0.69,2026-03-06 01:00:00+00:00,0.68,24707.51,0.04,7172.25,0.02,0.45,0.51,0.35
130,0x889e7f0464c72eb8cda1525ebc12b6aaba9d09e0,9.29,1732,237,12983350.38,608188.84,3590.42,0.37,0.22,0.06,2025-07-05 09:00:00+00:00,0.10,143594.34,0.24,50813.48,14.15,0.05,0.01,0.00
9,0x8861f0bb5e0c19474ba73beeadc13ed8915beed6,16.03,325,17,1361177.08,556342.53,129122.41,0.48,0.45,0.37,2025-05-29 11:35:00+00:00,1.05,25732.59,0.05,499.11,0.00,0.41,0.23,0.24
66,0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292,5.38,3478,549,9872547.92,356035.39,116014.93,0.22,0.12,0.01,2025-07-25 03:30:00+00:00,0.11,56515.67,0.16,35000.28,0.30,0.04,0.33,0.04
37,0xdfe3fedc5c7679be42c3d393e99d4b55247b73c4,9.91,1834,161,2472889.12,299573.49,52980.57,0.19,0.19,0.25,2025-09-05 16:50:00+00:00,0.42,34988.85,0.12,5532.21,0.10,0.12,0.18,0.07
96,0xee50a31c3f5a7c77824b12a941a54388a2827ed6,9.26,651,30,953234.83,293613.57,8262.76,0.24,0.24,0.41,2025-11-17 19:05:00+00:00,0.58,13550.70,0.05,2863.46,0.35,0.31,0.03,0.02
54,0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,10.90,5378,1046,3387993.42,278882.00,72383.65,0.40,0.08,0.06,2025-08-19 10:00:00+00:00,0.18,67467.21,0.24,35112.36,0.49,0.08,0.26,0.05
81,0x5739ddf8672627ce076eff5f444610a250075f1a,4.69,2252,94,2259574.03,254193.40,48981.69,0.15,0.13,0.15,2025-07-24 11:35:00+00:00,0.13,46039.12,0.18,34489.51,0.70,0.11,0.19,0.02
0,0x67327ef4880d4050ead195ca23203c61d8c1d7f0,14.22,130,9,78285.70,180156.86,32267.53,0.50,0.47,5.67,2026-02-27 00:22:30+00:00,7.91,3793.94,0.02,3925.98,0.12,2.30,0.18,1.42


In [30]:
selected_wallets = wallet_cohorts["multi_filter"]
selected_wallets.to_parquet(WORKSPACE_DIR / "selected_wallets_v2.parquet", index=False)

In [31]:
WORKSPACE_DIR/"selected_wallets_v2.parquet"

PosixPath('../../data/trade_signals_workspace_v2/selected_wallets_v2.parquet')

## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [32]:
# from polymarket_analysis.wallet_selection.selector import build_strategies_from_sweep
# from polymarket_analysis.strategy.registry import save_strategy

# all_strategies = build_strategies_from_sweep(
#     wallet_cohorts=wallet_cohorts,
#     signal_threshold=SIGNAL_THRESHOLD,
#     selection_metric=BEST_SELECTION_METRIC,
#     top_n=BEST_TOP_N,
#     sweep_df=cohort_sweep,
#     extra_metadata={
#         "end_date_train": str(END_DATE_TRAIN),
#         "train_a_end_date": str(TRAIN_A_END_DATE),
#     },
# )

# for strategy in all_strategies:
#     parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
#     print(f"Saved [{strategy.strategy_id}]  wallets={len(strategy.wallets):3d}  trigger={strategy.trigger.fn_ref.split('.')[-1]}")

# print(f"\nTotal strategies saved: {len(all_strategies)}")


## Strategy registry summary

In [33]:
# from polymarket_analysis.strategy.registry import load_all_strategies

# registry = load_all_strategies(WORKSPACE_DIR)
# summary_rows = []
# for sid, s in registry.items():
#     summary_rows.append({
#         "strategy_id": s.strategy_id,
#         "name": s.name,
#         "selection_method": s.selection_method,
#         "num_wallets": len(s.wallets),
#         "trigger_fn": s.trigger.fn_ref.split(".")[-1],
#         "threshold": s.trigger.params.get("threshold"),
#         "dynamic_sizing": s.trigger.params.get("dynamic_sizing"),
#     })

# pd.DataFrame(summary_rows)


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [34]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [35]:
df_fills = df_full.copy()
df_fills['copyable_pnl_exposure'] = np.where( 
    df_fills['copyable_qty'] > 0,
    df_fills['price'] * df_fills['copyable_qty'],
    np.nan
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

len(df_fills), len(df_fills[df_fills["dt"] < split_dt]), len(df_fills[df_fills["dt"] >= split_dt])

(5854490, 4525079, 1329411)

In [36]:
markets = df_fills[(df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))][['condition_id', 'tags', 'primary_tag']]
markets = markets.groupby('condition_id').agg(
    tags=('tags', 'first'),
    primary_tag=('primary_tag', 'first'),
).reset_index()
markets['has_mentions'] = markets['tags'].apply(lambda tags: 'Mentions' in tags)
mention_markets = set(markets[markets['has_mentions']]['condition_id'])
del markets
len(mention_markets)

8302

In [37]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    & (df_fills['primary_tag'] == 'Politics')
    & (~df_fills['condition_id'].isin(mention_markets))
    & (df_fills["dt"] >= split_dt)
    ].copy().reset_index(drop=True)

print(len(filter_fills))
filter_fills = filter_fills[filter_fills['copyable_qty'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head(2)

20211
9209


,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,...,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,copyable_pnl_exposure
0,0x04ec75cbba2a7f2407eb0a48a8468d25ee14580f,0x0de3f76fe9b60857833856bec4f30d815cdb0c283361...,2026-04-07 23:11:37+00:00,BUY,500.00,500.00,0.93,462.50,500.00,37.50,...,9.00,2026-04-10T00:00:00Z,US x Iran ceasefire by April 10?,"[Politics, Iran, Trump, Middle East, Geopoliti...",Politics,7996033923467324204298450566738037680866352090...,Yes,37.50,462.50,462.50
1,0x0ad7f3411bc87f0b5362155e638f04ef05700971,0x05f40f8a71edbdac0b4e2803970178ea7883bd9c6d24...,2026-04-19 23:19:58+00:00,BUY,7.27,7.27,0.07,0.51,0.00,-0.51,...,2.00,2026-04-30T00:00:00Z,Will Russia enter Ternuvate again by April 30?,"[Politics, Ukraine, Geopolitics, Ukraine Map]",Politics,7826800637181705094992541697034822732106425369...,Yes,-0.51,0.51,0.51


In [38]:
pd.set_option('display.max_colwidth', 100)

In [39]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor('5min')

MAX_EXPOSURE = 100

df = filter_fills.groupby(['bucket', 'condition_id', 'wallet', 'side']).agg(
    copyable_pnl_exposure=('copyable_pnl_exposure', 'sum'),
    total_copyable_qty=('copyable_qty', 'sum'),
    trade_pnl =('trade_pnl', 'sum'),
    copyable_pnl = ('copyable_pnl', 'sum'),
    trades=('condition_id', 'count'),
    copyable_qty=('copyable_qty', 'sum'),
    wallets=('wallet', lambda x: x.nunique()),
).reset_index()

scale = np.minimum(1, MAX_EXPOSURE / df["copyable_pnl_exposure"].replace(0, np.nan))

df = df.assign(
    scaled_copyable_pnl_exposure = df["copyable_pnl_exposure"] * scale,
    scaled_copyable_pnl = df["copyable_pnl"] * scale,
    scaled_copyable_qty = df["copyable_qty"] * scale,
)

df.head(10)

,bucket,condition_id,wallet,side,copyable_pnl_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,copyable_qty,wallets,scaled_copyable_pnl_exposure,scaled_copyable_pnl,scaled_copyable_qty
0,2026-03-16 00:15:00+00:00,0x84dafaed1275f878da543e1ae4e89634c70ead7ba43d4d067512ae5dcaeb0b1f,0xdf409ebc2cda86c98f3abf7666ff4bdc01ff38b8,BUY,67.00,100.00,33.00,33.00,1,100.00,1,67.00,33.00,100.00
1,2026-03-16 00:25:00+00:00,0xf74a25ca00930e1b6bf444fcff9ded49cd7765d6e198a9af9c036688edb5a4ed,0xdf409ebc2cda86c98f3abf7666ff4bdc01ff38b8,BUY,11.61,19.68,-11.61,-11.61,1,19.68,1,11.61,-11.61,19.68
2,2026-03-16 01:25:00+00:00,0x5a200d7d560169d60dc82cd16bb14c16f36f029fdf609dcb92d06a554f9f0fe1,0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,BUY,71.70,80.57,11.00,8.86,1,80.57,1,71.70,8.86,80.57
3,2026-03-16 01:25:00+00:00,0x6aa39e488afd5dd8e1cd995258c28c3c485dbc4d467d4eb4d94fc6f30e172833,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,BUY,68.21,94.74,-72.00,-68.21,1,94.74,1,68.21,-68.21,94.74
4,2026-03-16 01:35:00+00:00,0x7cb525e831729325d651017f81cbcb6f1adde5011c7b2283babea00b4ae93ae7,0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007,BUY,965.00,1000.00,35.00,35.00,2,1000.00,1,100.00,3.63,103.63
5,2026-03-16 02:10:00+00:00,0x84dafaed1275f878da543e1ae4e89634c70ead7ba43d4d067512ae5dcaeb0b1f,0xdf409ebc2cda86c98f3abf7666ff4bdc01ff38b8,BUY,50.16,85.02,34.86,34.86,1,85.02,1,50.16,34.86,85.02
6,2026-03-16 02:15:00+00:00,0x84dafaed1275f878da543e1ae4e89634c70ead7ba43d4d067512ae5dcaeb0b1f,0x41583f2efc720b8e2682750fffb67f2806fece9f,BUY,126.14,229.35,180.00,103.21,1,229.35,1,100.00,81.82,181.82
7,2026-03-16 03:00:00+00:00,0x773abaa5fe55e5cde51a261f444b7921652a4e059ead6b3be9fe56499c2d4609,0x60a92c8620846d81f5ea17b0564e0d4b7c545a71,BUY,10.00,13.33,-1000.00,-10.00,1,13.33,1,10.00,-10.00,13.33
8,2026-03-16 03:15:00+00:00,0x705479cc3e7cf2907997c62b4245356c48b5133f7d82a47829a2354dd77a7c51,0x60a92c8620846d81f5ea17b0564e0d4b7c545a71,BUY,30.88,37.84,276.00,6.96,1,37.84,1,30.88,6.96,37.84
9,2026-03-16 04:45:00+00:00,0x9ca29ccd6068cbaff7b108e3b32d90699bdd1afd139fb71ec5fba5e258b05a79,0xdf409ebc2cda86c98f3abf7666ff4bdc01ff38b8,BUY,24.60,30.00,5.40,5.40,1,30.00,1,24.60,5.40,30.00


In [40]:
df = (df.merge(mdf, on='condition_id', how='inner')
    .sort_values(['bucket', 'copyable_pnl_exposure'], ascending=[True, False])
    .reset_index(drop=True)
)
df['end_date_iso'] = pd.to_datetime(df['end_date_iso'], utc=True)
df = df[df['end_date_iso'] > split_dt].copy().reset_index(drop=True)

In [41]:
df_plot = df.groupby('end_date_iso').agg(
    scaled_copyable_pnl=('scaled_copyable_pnl', 'sum'),
    ending_exposure=('scaled_copyable_pnl_exposure', 'sum'),
).reset_index()

df.sort_values('bucket', inplace=True)
df['scaled_copyable_pnl_cumsum'] = df['scaled_copyable_pnl'].cumsum()

df_pnl = df[['bucket', 'scaled_copyable_pnl_cumsum']].copy()

# exposure ends at EOD of end_date, so shift by 24h
df_exposure = df_plot[['end_date_iso', 'ending_exposure']].copy()
df_exposure['end_date_iso'] = pd.to_datetime(df_exposure['end_date_iso']) + pd.Timedelta(days=1)
df_exposure.rename(columns={'end_date_iso': 'dt', 'ending_exposure': 'exposure_delta'}, inplace=True)

resolution_exposure = df_exposure.groupby('dt').agg(exposure_delta=('exposure_delta', 'sum')).reset_index()
resolution_exposure['exposure_delta'] = resolution_exposure['exposure_delta'] * -1

buy_exposure = df[['bucket', 'scaled_copyable_pnl_exposure']].rename(columns={'bucket': 'dt', 'scaled_copyable_pnl_exposure': 'exposure_delta'})

df_exposure = ( pd.concat([resolution_exposure, buy_exposure], ignore_index=True)
    .reset_index(drop=True)
)

df_exposure.sort_values('dt', inplace=True)
df_exposure['exposure'] = df_exposure['exposure_delta'].cumsum()

# df_plot['end_date_iso'] = pd.to_datetime(df_plot['end_date_iso']) + pd.Timedelta(days=1)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_exposure['dt'],
    y=df_exposure['exposure'],
    mode='lines',
    name='exposure'
))

fig.add_trace(go.Scatter(
    x=df_pnl['bucket'],
    y=df_pnl['scaled_copyable_pnl_cumsum'],
    mode='lines',
    name='scaled_copyable_pnl_cumsum'
))

fig.show()

In [42]:
import plotly.express as px

# Aggregate by bucket
bucket_pnl = df.groupby('bucket')['scaled_copyable_pnl'].sum().reset_index()
bucket_pnl['scaled_copyable_pnl'] = bucket_pnl['scaled_copyable_pnl'].cumsum()

# Plot with Plotly
fig = px.line(
    bucket_pnl,
    x='bucket',
    y='scaled_copyable_pnl',
    title='Total Scaled Copyable PnL by Time Bucket',
    labels={'bucket': 'Time Bucket', 'scaled_copyable_pnl': 'Scaled Copyable PnL'}
)
fig.show()

In [43]:
( 
    df.groupby('condition_id').agg(
        total_copyable_exposure=('copyable_pnl_exposure', 'sum'),
        total_copyable_qty=('copyable_qty', 'sum'),
        trade_pnl =('trade_pnl', 'sum'),
        copyable_pnl = ('copyable_pnl', 'sum'),
        trades=('condition_id', 'count'),
        wallets=('wallet', lambda x: x.nunique()),
 ).sort_values('copyable_pnl', ascending=False).head(10)
)

,total_copyable_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,wallets
condition_id,,,,,,
0x4c5701bcde0b8fb7d7f48c8e9d20245a6caa58c61a77f981fad98f2bfa0b1bc7,408049.10,793611.09,100993.23,98066.63,228,37
0x8183a221622f2d5d2b46da86c80d55a61f43d35829589e03c1efae9d072638c5,205674.52,311091.76,-54000.44,40866.36,305,32
0x7cb525e831729325d651017f81cbcb6f1adde5011c7b2283babea00b4ae93ae7,525371.56,567292.22,42040.63,21857.92,138,34
0x5c2e6aef8af5931e9bfa3750364626d754531d2fada2885d45c356b175962a25,61144.11,94767.73,26255.49,19479.11,148,23
0x9967297556e4fd7f7e1e688a2e44d6fdbc42b0700576a1e4e67aa37764ac640a,18569.63,34697.87,10687.51,11696.60,27,10
0xa6ddb7146f48a12dbf73456d654211b01d7493829932c31b7fe85d82120d338f,23641.20,38777.50,16491.82,8605.93,99,12
0x136f5a0c27a62cf9a2e40a4f48425e43d61b9571a53a2529372c0065f3218a73,12540.02,20507.36,28016.54,7967.34,23,4
0xd73f60114a0e7169a55082daef1228cb27fa50c939eea22cb0589f6bac6ce5d3,7717.99,15762.71,22650.69,7498.54,36,9
0x68beecc857017df8663fa36c0c86310dec7305aa88fe830c94651391314afc1c,69025.09,91938.98,5308.06,7304.36,68,18


In [44]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

else:
    print("PLOT_WALLET_PNL=False — skipping plots")